# YOLOv2 448 + horizontal flip and brightness jitter

**RECONSTRUCTED — this is not the original notebook.**

The original was never saved: the augmentation and schedule changes were
applied as patch cells inside a live Kaggle session, and the notebook was
downloaded before those cells were added. The run itself did happen, and
its weights are in `Data_new/runs/yolov2_rgb_448_aug/`.

This file reproduces what that session did, assembled from the session log.
Cells 0-4 are copied verbatim from `yolov2_rgb_448_kaggle.ipynb`, which is a
real saved notebook. The patch cell and the training command are the
reconstructed part.

**Result: val AP 0.711, test AP 0.634.** Validation improved over plain 448 (0.649); test got worse (0.711).

**Platform:** Kaggle, T4 GPU, internet on.
**Dataset:** `tensors_rgb_448_packed` uploaded as a Kaggle dataset, together
with `config.py`, `dataset.py`, `targets.py`, `train.py`, `evaluate.py`.


In [ ]:
!nvidia-smi
!ls -R /kaggle/input | head -20

In [ ]:
BASE = '/kaggle/input/datasets/bornamuzina/dataset448'
DATA = BASE + '/tensors_rgb_448_packed_k/tensors_rgb_448_packed'
!cp {BASE}/*.py /kaggle/working/
%cd /kaggle/working
!ls *.py

In [ ]:
!apt-get install -qq python3.11 python3.11-venv python3.11-dev > /dev/null 2>&1
!python3.11 -m venv /kaggle/working/akv
!/kaggle/working/akv/bin/pip install -q --upgrade pip
!/kaggle/working/akv/bin/pip install -q akida-models==1.14.2

In [ ]:
import re
src = open('config.py').read()
src = re.sub(r"^ROOT = Path\(.*?\)$", "ROOT = Path('/kaggle/working')", src, flags=re.M)
src = re.sub(r"^TENSOR_DIR = .*$", f"TENSOR_DIR = Path('{DATA}')", src, flags=re.M)
src = re.sub(r"^SPLITS_FILE = .*$", f"SPLITS_FILE = Path('{DATA}/splits.json')", src, flags=re.M)
src = re.sub(r"^RUNS_DIR = .*$", "RUNS_DIR = Path('/kaggle/working/runs')", src, flags=re.M)
open('config.py','w').write(src)

src = open('dataset.py').read()
src = src.replace(
    'if not npy.exists() or not meta_path.exists():',
    'npz = tensor_dir / f"{clip}_tensors.npz"\n    if not meta_path.exists() or (not npy.exists() and not npz.exists()):'
)
src = src.replace(
    'tensors = np.load(npy, mmap_mode="r")',
    'tensors = np.load(npy, mmap_mode="r") if npy.exists() else np.load(npz)["a"]'
)
open('dataset.py','w').write(src)

!grep -n "ROOT\|TENSOR_DIR\|SPLITS_FILE\|RUNS_DIR\|INPUT_SIZE\|^GRID" config.py

In [ ]:
!MPLBACKEND=Agg /kaggle/working/akv/bin/python -u dataset.py

## The patch

Applied to `train.py` in the running session. Two changes: define
`augment_batch`, then call it inside `run_epoch` for training batches only —
validation must stay fixed or the numbers stop being comparable between runs.

Horizontal flip mirrors the image and the box x coordinate. No vertical flip:
drones do not appear upside down against ground, so it would teach a
configuration that never occurs.

In [ ]:
aug = """

def augment_batch(images, boxes_list, size=None, rng=np.random):
    \"\"\"Horizontal flip and brightness jitter. Training only.\"\"\"
    if size is None:
        size = config.INPUT_SIZE

    out_images = []
    out_boxes = []

    for img, boxes in zip(images, boxes_list):
        img = np.asarray(img)
        boxes = list(boxes)

        if rng.random() < 0.5:
            img = img[:, ::-1]
            boxes = [(size - x - w, y, w, h) for (x, y, w, h) in boxes]

        if rng.random() < 0.5:
            gain = rng.uniform(0.7, 1.3)
            bias = rng.uniform(-25.0, 25.0)
            img = np.clip(img * gain + bias, 0.0, 255.0)

        out_images.append(img)
        out_boxes.append(boxes)

    return np.stack(out_images), out_boxes

"""

src = open('train.py').read()

# define the function just before the loss section
src = src.replace(
    '\n# ============================================================\n# LOSS\n',
    aug + '\n# ============================================================\n# LOSS\n'
)

# call it, training only
src = src.replace(
    '        x = to_uint8(images)\n        y = T.make_targets(boxes_list)',
    '        if training:\n            images, boxes_list = augment_batch(images, boxes_list)\n\n'
    '        x = to_uint8(images)\n        y = T.make_targets(boxes_list)'
)

open('train.py','w').write(src)

!grep -n "def augment_batch\|if training:" train.py

## Verify before training

The flip is the part that can silently produce wrong targets, so check the
box coordinate actually mirrors. x should alternate between 100 and 332
(448 − 100 − 16) and never go negative.

In [ ]:
test = """
import numpy as np, config, train as t
img = np.zeros((1, config.INPUT_SIZE, config.INPUT_SIZE, 3), np.float32)
img[0, 40:52, 100:116] = 200
boxes = [[(100.0, 40.0, 16.0, 12.0)]]
rng = np.random.RandomState(1)
for i in range(10):
    a, b = t.augment_batch(img, boxes, rng=rng)
    x, y, w, h = b[0][0]
    print(f"{i}: x={x:.0f}  pixels {a.min():.0f}-{a.max():.0f}")
"""
open('test_aug.py','w').write(test)

In [ ]:
!MPLBACKEND=Agg /kaggle/working/akv/bin/python test_aug.py

## Train

25 epochs rather than 15: the point of augmentation is that the model should
keep improving past the epoch 2 peak the un-augmented run hit.

In [ ]:
!MPLBACKEND=Agg /kaggle/working/akv/bin/python -u train.py --epochs 25 --lr 1e-3 --batch_size 32 --name full_rgb_448_aug

In [ ]:
!MPLBACKEND=Agg /kaggle/working/akv/bin/python -u evaluate.py --run full_rgb_448_aug --split validation

In [ ]:
!MPLBACKEND=Agg /kaggle/working/akv/bin/python -u evaluate.py --run full_rgb_448_aug --split test
!cd /kaggle/working && zip -qr full_rgb_448_aug.zip runs/full_rgb_448_aug
!ls -lh /kaggle/working/full_rgb_448_aug.zip

## Result

| | val | test |
|---|---|---|
| AP@0.5 | 0.711 | 0.634 |
| precision | 0.777 | 0.674 |
| recall | 0.794 | 0.731 |

Best epoch 9, val loss 1.681 — *worse* than the un-augmented run's 1.545, and
train recall still climbed to 0.969. The loss curve looked like a failure and
the validation AP was the best of any 448 run.

**The lesson: judge augmentation on AP, not on validation loss.** Loss is only
meaningful for comparing epochs within one run.

Validation improved (0.649 → 0.711) and test got worse (0.711 → 0.634), so the
result is not a clean win either way.